# Notebook 05: LLM Interview Assistant Evaluation

**Project:** `ofphil-classification-v2`  
**Phase:** 05 — Model Selection and Training (`06_llm_interview_assistant`)  
**Objective:** Establish a standardized, objective evaluation framework across all four candidate models without using classification metrics or fabricated benchmark scores.

---

## 1. Evaluation Methodology & Dimensions

The MSW Interview Assistant is evaluated across **9 qualitative and operational dimensions** on a standardized 1–5 rubric:

1. **Question Relevance**: Are suggested questions logically aligned with the active intake domain?
2. **Follow-Up Quality**: Do suggested follow-ups target genuine ambiguities in patient statements?
3. **Filipino / Taglish Comprehension**: Does the model accurately grasp colloquial phrasing (e.g. *"ambagan"*, *"namamasada"*, *"hulugan"*)?
4. **Conversation Consistency**: Does the model retain facts across multi-turn exchanges without contradiction?
5. **Structured Extraction Validity**: Does the model produce valid, Pydantic-parseable JSON conforming to schema?
6. **Missing-Information Handling**: Does the model correctly output `null` for unstated facts rather than inventing answers?
7. **Hallucination Rate**: Zero unsupported clinical facts, invented discounts, or unverified claims.
8. **Response Latency**: Seconds per generated follow-up response on local hardware.
9. **Resource Consumption**: Peak VRAM and System RAM footprint under Ollama.

*Crucial Governance Note: Traditional eligibility classification metrics (Accuracy, ROC-AUC, F1-Score) belong exclusively to the separate tabular models in Phase 05, NOT to the conversational LLM.*

In [ ]:
# Define standardized evaluation criteria and candidate list
import pandas as pd
import json
from pathlib import Path

EVALUATION_DIMENSIONS = [
    "Question Relevance (1-5)",
    "Follow-Up Quality (1-5)",
    "Taglish Comprehension (1-5)",
    "Conversation Consistency (1-5)",
    "Schema Extraction Validity (% Valid JSON)",
    "Missing-Information Handling (% Correct Nulls)",
    "Hallucination-Free Rate (% Clean)",
    "Avg Latency (seconds/turn)",
    "Peak VRAM Usage (GB)"
]

CANDIDATE_MODELS = [
    {"tag": "qwen3:4b", "name": "Qwen3 4B Instruct", "priority": "Initial Testing Candidate"},
    {"tag": "phi4-mini", "name": "Phi-4-mini 3.8B Instruct", "priority": "Lightweight Comparison"},
    {"tag": "qwen3:8b", "name": "Qwen3 8B Instruct", "priority": "Higher-Capacity Comparison"},
    {"tag": "gemma3:12b", "name": "Gemma 3 12B Instruct", "priority": "Optional Heavy Comparison"}
]

print(f"Evaluation framework ready across {len(EVALUATION_DIMENSIONS)} dimensions for {len(CANDIDATE_MODELS)} candidates.")

## 2. Standardized Synthetic Test Scenarios

We define 4 challenging synthetic interview test scenarios covering common casework challenges:
- **Scenario 1**: Informal variable daily wage (tricycle driver / street vendor).
- **Scenario 2**: Extended family household with multi-generational dependents.
- **Scenario 3**: High medical burden with incomplete PhilHealth / Malasakit documentation.
- **Scenario 4**: Vague or evasive response requiring gentle clarification.

In [ ]:
# Synthetic test scenarios dictionary
test_scenarios = [
    {
        "scenario_id": "SCEN-01-INFORMAL-WAGE",
        "title": "Informal Day Laborer with Fluctuating Income",
        "patient_statement": "Arawan po ang kita ko sa construction, minsan 450 sa isang araw pag may project pero nitong buwan dalawang beses lang ako natawagan dahil masakit ang mata ko.",
        "expected_unclarified_fields": ["monthly_family_income", "alternative_earners"],
        "evaluation_focus": "Does the model avoid setting income to zero and avoid hallucinating a monthly total?"
    },
    {
        "scenario_id": "SCEN-02-EXTENDED-FAMILY",
        "title": "Multi-Generational Shared Living",
        "patient_statement": "Nakatira po kami sa biyenan ko kasama ang pamilya ng hipag ko. Apat kaming mag-asawa at mga anak ko, pero pito kaming lahat sa bahay.",
        "expected_unclarified_fields": ["household_size", "dependent_count", "shared_expenses"],
        "evaluation_focus": "Does the model distinguish between nuclear family size and total household occupants?"
    },
    {
        "scenario_id": "SCEN-03-PARTIAL-SUBSIDY",
        "title": "Existing Assistance Guarantee from LGU",
        "patient_statement": "May binigay pong guarantee letter ang mayor na 5,000 pesos pero ang sabi sa ospital 25,000 daw ang kailangan sa lente at gamot.",
        "expected_unclarified_fields": ["pre_existing_assistance", "total_procedure_cost"],
        "evaluation_focus": "Does the model isolate the 5,000 LGU assistance without deducting it automatically as approved OFPHIL aid?"
    },
    {
        "scenario_id": "SCEN-04-VAGUE-OBLIGATIONS",
        "title": "Unspecified Household Utility Debts",
        "patient_statement": "Medyo malaki po ang utang namin sa kuryente at sari-sari store dahil sa gamot ng nanay ko pero hindi ko po eksaktong kabisado magkano.",
        "expected_unclarified_fields": ["monthly_utility_expense", "outstanding_debts"],
        "evaluation_focus": "Does the model propose gentle follow-up questions to identify approximate bills rather than leaving it blank or guessing?"
    }
]

print(f"Configured {len(test_scenarios)} standardized synthetic test scenarios.")

## 3. Comparative Evaluation Results Schema

The table below represents the empty results schema ready for manual benchmark population once all models are pulled and tested through Ollama on local hardware.

*Note: In accordance with project research integrity guidelines, no fabricated benchmark scores are inserted prior to actual execution.*

In [ ]:
# Empty comparative evaluation results table
eval_records = []
for c in CANDIDATE_MODELS:
    eval_records.append({
        "Model Candidate": c["name"],
        "Ollama Tag": c["tag"],
        "Priority": c["priority"],
        "Question Relevance (1-5)": None,
        "Follow-Up Quality (1-5)": None,
        "Taglish Comprehension (1-5)": None,
        "Consistency (1-5)": None,
        "Schema Validity (%)": None,
        "Missing Info Handling (%)": None,
        "Hallucination-Free (%)": None,
        "Avg Latency (s)": None,
        "Peak VRAM (GB)": None,
        "Evaluation Status": "Pending Manual Benchmark Execution"
    })

df_eval_schema = pd.DataFrame(eval_records)
print("Evaluation Table Schema (Awaiting Local Execution):")
print(df_eval_schema[["Model Candidate", "Ollama Tag", "Priority", "Evaluation Status"]].to_string(index=False))

# Export template to interim evaluation directory
OUT_DIR = Path.cwd().resolve()
while OUT_DIR.name != 'ofphil-classification-v2' and OUT_DIR.parent != OUT_DIR:
    OUT_DIR = OUT_DIR.parent
OUT_DIR = OUT_DIR / 'data' / 'interim' / 'evaluation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_eval_schema.to_csv(OUT_DIR / 'llm_interview_assistant_benchmark_schema.csv', index=False)
print(f"\nSaved empty benchmark schema to: {OUT_DIR / 'llm_interview_assistant_benchmark_schema.csv'}")